# nb4 — Phase 2 (Bước 1): Error analysis stratified trên TEST + adjudication pseudo-gold

Notebook **eval-only, không train, không cần GPU** (chạy notebook Kaggle loại **CPU** để tiết kiệm quota — `DESIGN.md` §7, quyết định Phase 2 23/09 trong `DESIGN.md` §12).

Phân tích **các prediction đã có** của nb3/nb3b (không chạy model) để trả lời 7 câu hỏi của Bước 1:

| # | Câu hỏi | Bảng |
|---|---|---|
| Q1 | Correction acc @TP trên test thấp (66–68%) — sai ở non-word hay real-word? | T1 |
| Q2 | Model bỏ sót lỗi khi câu dày lỗi? | T2 |
| Q3 | FP deletion chiếm bao nhiêu % FP (đầu vào cho Bước 2 post-processing)? | T3 |
| Q4 | Khi sửa sai nội dung ở vị trí detect đúng, model sửa thành gì? | T4 |
| Q5 | FN là copy thuần hay model edit lung tung? | T5 |
| Q6 | Có bias theo vị trí trong câu / độ dài câu? | T6 |
| Q7 | Pseudo-gold của test nhiễu bao nhiêu? | Adjudication 100 câu suspect |

**Input** (attach Kaggle Input — output của các notebook trước):
- Bắt buộc: `test_aligned.jsonl` (nb1), `vsec_val.jsonl` (nb0), `syllable_table.json` (nb2), `eval_report.json` (nb3), `predictions_{val,test}_run{1,2}.jsonl` (nb3).
- Tùy chọn: `predictions_{val,test}_zeroshot.jsonl` + `zeroshot_eval_report.json` (nb3b — thiếu thì bỏ cột Zero-shot), `adjudication_filled.json` (bản đã điền nhãn A/B/C/D — xem §6).

**Output** (ghi `/kaggle/working`):
- `error_analysis_report.json` — toàn bộ bảng T1–T7 + consistency check + adjudication.
- `adjudication_samples.json` — 100 mẫu suspect + rubric, để điền nhãn thủ công.

**Quy tắc chống leakage (`DESIGN.md` §9)**: notebook chỉ **đọc** val/test để đánh giá; không nạp train cho bất kỳ thống kê nào; bảng âm tiết là nguồn công khai độc lập (nb2); nhãn adjudication là đo lường của con người — chỉ dùng để ước lượng nhiễu pseudo-gold + sensitivity, **không fit tham số model/pipeline**.

**Vận hành trên Kaggle (~vài phút, CPU)**:
1. Tạo Kaggle notebook mới, **không bật GPU** (chọn CPU để không tốn quota).
2. Attach Input: dataset chứa các file nêu trên (output nb0→nb3, tùy chọn nb3b).
3. Upload notebook, Run All. Kiểm tra **§4 CONSISTENCY CHECK phải PASS** trước khi tin các bảng.
4. Tải `adjudication_samples.json`, điền `label` ∈ {A,B,C,D} cho 100 mẫu (rubric ở §6), đổi tên `adjudication_filled.json`, upload lại làm Input, chạy lại từ đầu để có ước lượng nhiễu + sensitivity.


In [1]:
import os
import json
import math
import random
import datetime
import unicodedata
import collections
from pathlib import Path

SEED = 42
ADJ_N = 100  # số mẫu suspect adjudicate thủ công (plan Bước 1 Phase 2)

# Buckets phân tích — mọi hằng số gom một chỗ (DESIGN.md §7)
DENSITY_BINS = ['1', '2', '3', '>=4']  # theo SỐ GOLD EDIT BLOCK/câu (khớp thống kê nb1)
LEN_BUCKETS = [('<15', 0, 15), ('15-30', 15, 30), ('30-50', 30, 50), ('>=50', 50, 10 ** 9)]
RELPOS_BUCKETS = [('0-25%', 0.0, 0.25), ('25-50%', 0.25, 0.5), ('50-75%', 0.5, 0.75), ('75-100%', 0.75, 1.0001)]
ADJ_RATIO_STRATA = [('0.3-0.4', 0.3, 0.4), ('0.4-0.5', 0.4, 0.5), ('>=0.5', 0.5, 10.0)]

PRED_FILES = {
    'val_run1': 'predictions_val_run1.jsonl',
    'val_run2': 'predictions_val_run2.jsonl',
    'test_run1': 'predictions_test_run1.jsonl',
    'test_run2': 'predictions_test_run2.jsonl',
}
PRED_FILES_OPT = {
    'val_zeroshot': 'predictions_val_zeroshot.jsonl',
    'test_zeroshot': 'predictions_test_zeroshot.jsonl',
}
REPORT_NB3 = 'eval_report.json'
REPORT_NB3B = 'zeroshot_eval_report.json'

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

REQUIRED_FILES = ['test_aligned.jsonl', 'vsec_val.jsonl', 'syllable_table.json', REPORT_NB3] + sorted(PRED_FILES.values())


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
    found = {}
    for p in candidates:
        if p.name not in found:
            found[p.name] = p
    missing = [f for f in REQUIRED_FILES if f not in found]
    if missing:
        raise FileNotFoundError(
            f'Thiếu các tệp đầu vào bắt buộc: {missing}. '
            'Hãy attach Input: output nb0 (vsec_val), nb1 (test_aligned), nb2 (syllable_table), '
            f'nb3 (predictions run1/run2 + {REPORT_NB3}). Zeroshot (nb3b) là tùy chọn.')
    return found


INPUT_FILES = find_required_inputs()
print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k in sorted(INPUT_FILES):
    req = k in REQUIRED_FILES
    print(f'  {k:36s}: {INPUT_FILES[k]}' + ('' if req else '  (tùy chọn)'))
missing_opt = [f for f in sorted(PRED_FILES_OPT.values()) + [REPORT_NB3B, 'adjudication_filled.json'] if f not in INPUT_FILES]
if missing_opt:
    print(f'Ghi chú: thiếu input tùy chọn {missing_opt} → bỏ qua tính năng tương ứng.')
print(f'Notebook: nb4-error-analysis · eval-only (không model, không GPU) · seed={SEED}')


=== TỰ DÒ INPUT HOÀN TẤT ===
  __huggingface_repos__.json          : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/__huggingface_repos__.json  (tùy chọn)
  __output__.json                     : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/__output__.json  (tùy chọn)
  adapter_config.json                 : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/checkpoints_run1_pure/checkpoint-655/adapter_config.json  (tùy chọn)
  adjudication_filled.json            : /kaggle/input/datasets/cquangnguynl/nb4-adjudication-filled/adjudication_filled.json  (tùy chọn)
  align_report.json                   : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/align_report.json  (tùy chọn)
  eval_report.json                    : /kaggle/input/notebooks/cquangnguynl/nb3-baseline-train-eval/eval_report.json
  manifest.json                       : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/manifest.json  (tùy chọn)
  noise_model.json                    : /kaggle/input/not

In [2]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)


SHARED_CELLS_VERSION: align-v1


<>:14: SyntaxWarning: invalid escape sequence '\w'
<>:14: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_58/4204501438.py:14: SyntaxWarning: invalid escape sequence '\w'
  """NFC + tách token: run chữ/số liền kề (\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""


## §2. Phân tích per-sentence — tái tạo ĐÚNG semantics của `evaluate_predictions` (nb3)

`analyze_sentence` trả về 1 summary + 3 list events mức vị trí:
- **TP event** (1/vị trí): `pos, src_tok, gold_tgt, pred_tgt, correct, err_cat (nonword/realword), density_bin, len_bucket, rel_pos`.
- **FN event** (1/vị trí): như trên + `model_edit_count_in_sent, min_dist_to_model_edit` (phân tích Q5).
- **FP event** (1/vị trí, gắn block type của prediction): `block_type, punct_only, src_tok, pred_str, sentence_clean` (Q3, Q7-T7).

Invariants bắt buộc (khớp nb3 tuyệt đối, sẽ được §4 đối chiếu số với `eval_report.json`):
- TP = gold∩pred positions; FP = pred−gold; FN = gold−pred (mức **vị trí nguồn**, block `insert` không chiếm vị trí).
- Correction đúng khi `pred_str.lower() == target_str.lower()` (target/pred = tgt_tokens join space của block).
- Over-correction denominator = số word token − gold positions; câu sạch = 0 gold position; stratified = `src_tok.lower() ∉ SYLL_SET → nonword`.
- Roundtrip invariant `apply_edit_blocks(src, blocks) == tgt` asserted 100% câu (align-v1, nb1).


In [3]:
def _bucket_of(value, buckets):
    for name, lo, hi in buckets:
        if lo <= value < hi:
            return name
    return buckets[-1][0]


def len_bucket(n_tokens):
    return _bucket_of(n_tokens, LEN_BUCKETS)


def rel_pos_bucket(pos, n_tokens):
    if n_tokens <= 0:
        return RELPOS_BUCKETS[0][0]
    return _bucket_of(pos / n_tokens, RELPOS_BUCKETS)


def err_cat_of(tok, syll_set):
    """Non-word nếu âm tiết nguồn ở vị trí lỗi không có trong bảng — đúng quy ước stratified của nb3."""
    if syll_set is None:
        return None
    return 'nonword' if tok.lower() not in syll_set else 'realword'


def analyze_sentence(sent_id, rec, pred, syll_set=None):
    src_toks = canon_tokenize(rec['text'])
    gold_toks = canon_tokenize(rec['corrected_text'])
    pred_toks = canon_tokenize(pred)

    gold_blocks = extract_edit_blocks(src_toks, gold_toks, levenshtein_opcodes(src_toks, gold_toks))
    pred_blocks = extract_edit_blocks(src_toks, pred_toks, levenshtein_opcodes(src_toks, pred_toks))

    assert apply_edit_blocks(src_toks, gold_blocks) == gold_toks, f'sent {sent_id}: roundtrip gold FAIL'
    assert apply_edit_blocks(src_toks, pred_blocks) == pred_toks, f'sent {sent_id}: roundtrip pred FAIL'

    gold_pos_map = {}
    pred_pos_map = {}
    pred_block_of_pos = {}
    for b in gold_blocks:
        if b['src_span'][0] < b['src_span'][1]:
            target_str = ' '.join(b['tgt_tokens'])
            for p in range(b['src_span'][0], b['src_span'][1]):
                gold_pos_map[p] = target_str
    for b in pred_blocks:
        if b['src_span'][0] < b['src_span'][1]:
            pred_str = ' '.join(b['tgt_tokens'])
            for p in range(b['src_span'][0], b['src_span'][1]):
                pred_pos_map[p] = pred_str
                pred_block_of_pos[p] = b

    gold_positions = set(gold_pos_map)
    pred_positions = set(pred_pos_map)
    tp_pos = gold_positions & pred_positions
    fp_pos = pred_positions - gold_positions
    fn_pos = gold_positions - pred_positions

    word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
    clean_tokens = len(word_token_positions - gold_positions)
    is_clean = len(gold_positions) == 0
    clean_preserved = is_clean and len(pred_positions) == 0

    n = len(src_toks)
    lb = len_bucket(n)
    db = str(len(gold_blocks)) if len(gold_blocks) < 4 else '>=4'

    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0},
    }
    tp_events, fn_events, fp_events = [], [], []

    for p in sorted(tp_pos):
        target_str = gold_pos_map[p]
        pred_str = pred_pos_map[p]
        correct = pred_str.lower() == target_str.lower()
        cat = None
        if syll_set is not None and p < n:
            cat = err_cat_of(src_toks[p], syll_set)
            stratified[cat]['gold'] += 1
            stratified[cat]['detected'] += 1
            if correct:
                stratified[cat]['corrected'] += 1
        tp_events.append({
            'sent_id': sent_id, 'pos': p,
            'src_tok': src_toks[p] if p < n else None,
            'gold_tgt': target_str, 'pred_tgt': pred_str, 'correct': correct,
            'err_cat': cat, 'density_bin': db, 'len_bucket': lb,
            'rel_pos': rel_pos_bucket(p, n),
        })

    for p in sorted(fn_pos):
        cat = None
        if syll_set is not None and p < n:
            cat = err_cat_of(src_toks[p], syll_set)
            stratified[cat]['gold'] += 1
        if pred_positions:
            min_dist = min(abs(p - q) for q in pred_positions)
        else:
            min_dist = None
        fn_events.append({
            'sent_id': sent_id, 'pos': p,
            'src_tok': src_toks[p] if p < n else None,
            'gold_tgt': gold_pos_map[p], 'err_cat': cat,
            'density_bin': db, 'len_bucket': lb, 'rel_pos': rel_pos_bucket(p, n),
            'model_edit_count_in_sent': len(pred_blocks),
            'min_dist_to_model_edit': min_dist,
        })

    for p in sorted(fp_pos):
        b = pred_block_of_pos[p]
        fp_events.append({
            'sent_id': sent_id, 'pos': p,
            'src_tok': src_toks[p] if p < n else None,
            'block_type': b['type'], 'punct_only': b['punct_only'],
            'src_str': ' '.join(b['src_tokens']), 'pred_str': pred_pos_map[p],
            'sentence_clean': is_clean, 'density_bin': db,
            'len_bucket': lb, 'rel_pos': rel_pos_bucket(p, n),
        })

    summary = {
        'sent_id': sent_id,
        'tp': len(tp_pos), 'fp': len(fp_pos), 'fn': len(fn_pos),
        'correct_at_tp': sum(1 for e in tp_events if e['correct']),
        'clean_tokens': clean_tokens,
        'is_clean': is_clean, 'clean_preserved': clean_preserved,
        'n_gold_blocks': len(gold_blocks),
        'density_bin': db, 'len_bucket': lb,
        'stratified': stratified,
    }
    return summary, tp_events, fn_events, fp_events


def aggregate_sets(analyzed):
    """Gom kết quả analyze_sentence → tổng thể. Công thức khớp nguyên văn evaluate_predictions (nb3)."""
    tp = fp = fn = correct = clean_tokens = 0
    clean_total = clean_kept = 0
    stratified = {c: collections.Counter() for c in ('nonword', 'realword')}
    tp_events, fn_events, fp_events = [], [], []
    density = collections.Counter()
    for s, tps, fns, fps in analyzed:
        tp += s['tp']
        fp += s['fp']
        fn += s['fn']
        correct += s['correct_at_tp']
        clean_tokens += s['clean_tokens']
        if s['is_clean']:
            clean_total += 1
        if s['clean_preserved']:
            clean_kept += 1
        density[s['density_bin']] += 1
        for c in stratified:
            stratified[c].update(s['stratified'][c])
        tp_events += tps
        fn_events += fns
        fp_events += fps
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        'tp': tp, 'fp': fp, 'fn': fn, 'correct_at_tp': correct,
        'precision': precision, 'recall': recall, 'f1': f1,
        'corr_acc': correct / tp if tp > 0 else 0.0,
        'overcorr': fp / clean_tokens if clean_tokens > 0 else 0.0,
        'clean_tokens': clean_tokens,
        'clean_total': clean_total, 'clean_kept': clean_kept,
        'clean_retention': clean_kept / clean_total if clean_total > 0 else 1.0,
        'stratified': stratified, 'density': density,
        'tp_events': tp_events, 'fn_events': fn_events, 'fp_events': fp_events,
    }


print('Đã định nghĩa analyze_sentence + aggregate_sets (khớp semantics evaluate_predictions của nb3).')


Đã định nghĩa analyze_sentence + aggregate_sets (khớp semantics evaluate_predictions của nb3).


In [4]:
# Sanity test trên case nhân tạo — PASS bắt buộc trước khi chạy dữ liệu thật
S1 = {'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}

# Case 1: TP + FN (khớp sanity của nb3)
s, tps, fns, fps = analyze_sentence(0, S1, 'học sinh đi hoc')
assert s['tp'] == 1 and s['fn'] == 1 and s['fp'] == 0 and s['correct_at_tp'] == 1
assert len(tps) == 1 and tps[0]['gold_tgt'] == 'sinh' and tps[0]['pred_tgt'] == 'sinh' and tps[0]['correct']
assert len(fns) == 1 and fns[0]['src_tok'] == 'hoc'
assert fns[0]['model_edit_count_in_sent'] == 1 and fns[0]['min_dist_to_model_edit'] == 2
assert s['density_bin'] == '2' and s['len_bucket'] == '<15' and tps[0]['rel_pos'] == '25-50%'

# Case 2: FP deletion phá câu sạch (Q3/T7)
s, tps, fns, fps = analyze_sentence(1, {'text': 'học sinh đi học', 'corrected_text': 'học sinh đi học'}, 'học đi học')
assert s['is_clean'] and not s['clean_preserved'] and s['fp'] == 1 and s['tp'] == 0 and s['fn'] == 0
assert len(fps) == 1 and fps[0]['block_type'] == 'delete' and fps[0]['pred_str'] == '' and fps[0]['sentence_clean']

# Case 3: FP substitute (token thường) + FP punct_only
s, tps, fns, fps = analyze_sentence(2, {'text': 'Hòa bình :', 'corrected_text': 'Hòa bình :'}, 'hòa bình .')
assert s['is_clean'] and s['fp'] == 2
assert sorted(e['block_type'] for e in fps) == ['substitute', 'substitute']
assert sum(1 for e in fps if e['punct_only']) == 1

# Case 4: gold merge block chiếm 2 vị trí nguồn → 2 TP khi model sửa đúng
s, tps, fns, fps = analyze_sentence(3, {'text': 'Tuy nh iên ,', 'corrected_text': 'Tuy nhiên ,'}, 'Tuy nhiên ,')
assert s['tp'] == 2 and s['correct_at_tp'] == 2 and s['fn'] == 0
assert tps[0]['gold_tgt'] == 'nhiên' and tps[1]['gold_tgt'] == 'nhiên'

# Case 5: như Case 4 nhưng model sửa sai nội dung → correct_at_tp = 0
s, tps, fns, fps = analyze_sentence(4, {'text': 'Tuy nh iên ,', 'corrected_text': 'Tuy nhiên ,'}, 'Tuy nhien ,')
assert s['tp'] == 2 and s['correct_at_tp'] == 0
assert all(not e['correct'] for e in tps) and tps[0]['pred_tgt'] == 'nhien'

# Case 6: stratified nonword/realword + aggregate (khớp logic nb3: gold = TP + FN)
_syll = {'học', 'sinh', 'đi'}
s, tps, fns, fps = analyze_sentence(5, S1, 'học sinh đi hoc', _syll)
assert s['stratified']['nonword'] == {'gold': 2, 'detected': 1, 'corrected': 1}
assert s['stratified']['realword'] == {'gold': 0, 'detected': 0, 'corrected': 0}
_agg = aggregate_sets([(s, tps, fns, fps)])
assert _agg['tp'] == 1 and _agg['fn'] == 1 and _agg['fp'] == 0
assert _agg['corr_acc'] == 1.0 and _agg['f1'] == 2 / 3
assert _agg['stratified']['nonword']['gold'] == 2

# Case 7: bucket helpers
assert len_bucket(14) == '<15' and len_bucket(15) == '15-30' and len_bucket(49) == '30-50' and len_bucket(50) == '>=50'
assert rel_pos_bucket(0, 10) == '0-25%' and rel_pos_bucket(9, 10) == '75-100%'
assert err_cat_of('sanh', _syll) == 'nonword' and err_cat_of('học', _syll) == 'realword'

print('Sanity check analyze_sentence/aggregate_sets PASS 100% (7 case nhân tạo).')


Sanity check analyze_sentence/aggregate_sets PASS 100% (7 case nhân tạo).


## §3. Nạp dữ liệu + chạy phân tích trên các bộ predictions có sẵn

- Bắt buộc: `{val,test} × {run1, run2}` · tùy chọn: zeroshot (nb3b).
- Chống ghép lệch Input: so khớp `text` từng dòng giữa records và predictions — lệch là fail ngay.
- Đối chiếu thông tin: số gold block tính tại chỗ vs `error_count` do nb1 ghi trong `test_aligned.jsonl`.


In [5]:
val_records = load_jsonl(INPUT_FILES['vsec_val.jsonl'])
test_records = load_jsonl(INPUT_FILES['test_aligned.jsonl'])

SYLL_SET = set(json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))['entries'])
print(f'SYLL_SET: {len(SYLL_SET)} âm tiết (nguồn công khai độc lập — nb2).')

_tver = test_records[0].get('shared_cells_version')
assert _tver == SHARED_CELLS_VERSION, (
    f'test_aligned.jsonl: shared_cells_version={_tver!r} != {SHARED_CELLS_VERSION!r} — phiên bản aligner lệch, không so sánh được!')


def load_pred_set(records, pred_file, set_name):
    preds_raw = load_jsonl(INPUT_FILES[pred_file])
    assert len(preds_raw) == len(records), f'{set_name}: {len(preds_raw)} predictions vs {len(records)} records'
    preds = []
    for i, (r, pr) in enumerate(zip(records, preds_raw)):
        if nfc_normalize(pr['text']) != nfc_normalize(r['text']):
            raise ValueError(f'{set_name} dòng {i}: predictions không khớp records — ghép sai phiên bản Input?')
        preds.append(pr['prediction'])
    return preds


def run_analysis(records, preds, set_name, syll_set):
    analyzed = [analyze_sentence(i, r, p, syll_set) for i, (r, p) in enumerate(zip(records, preds))]
    agg = aggregate_sets(analyzed)
    tp_s = str(agg['tp'])
    fp_s = str(agg['fp'])
    fn_s = str(agg['fn'])
    print(f'{set_name:14s}: {len(records)} câu · TP={tp_s} FP={fp_s} FN={fn_s} · '
          f'F1={agg["f1"]:.2%} · CorrAcc={agg["corr_acc"]:.2%} · Over-corr={agg["overcorr"]:.2%}')
    return analyzed, agg


ANALYSIS = {}
REC_PREDS = {}
ALL_SETS = dict(PRED_FILES)
ALL_SETS.update({k: v for k, v in PRED_FILES_OPT.items() if v in INPUT_FILES})
for set_name, pred_file in ALL_SETS.items():
    records = val_records if set_name.startswith('val') else test_records
    preds = load_pred_set(records, pred_file, set_name)
    ANALYSIS[set_name] = run_analysis(records, preds, set_name, SYLL_SET)
    REC_PREDS[set_name] = (records, preds)

_mism = [i for i, r in enumerate(test_records)
         if r.get('error_count') is not None
         and r['error_count'] != ANALYSIS['test_run1'][0][i][0]['n_gold_blocks']]
_msg = f'{len(_mism)}/{len(test_records)} lệch'
print(f'Đối chiếu gold block (align tại chỗ vs error_count nb1): {_msg}')
if _mism:
    _ex = [(i, test_records[i].get('error_count'), ANALYSIS['test_run1'][0][i][0]['n_gold_blocks']) for i in _mism[:5]]
    print('  5 ví dụ đầu (idx, error_count nb1, tính tại chỗ):', _ex)


SYLL_SET: 7884 âm tiết (nguồn công khai độc lập — nb2).
val_run1      : 927 câu · TP=826 FP=200 FN=317 · F1=76.16% · CorrAcc=84.75% · Over-corr=0.75%
val_run2      : 927 câu · TP=809 FP=215 FN=334 · F1=74.67% · CorrAcc=86.53% · Over-corr=0.81%
test_run1     : 5983 câu · TP=17113 FP=2780 FN=7772 · F1=76.43% · CorrAcc=66.20% · Over-corr=1.82%
test_run2     : 5983 câu · TP=18680 FP=2872 FN=6205 · F1=80.45% · CorrAcc=67.94% · Over-corr=1.88%
Đối chiếu gold block (align tại chỗ vs error_count nb1): 0/5983 lệch


## §4. CONSISTENCY CHECK — bắt buộc PASS trước khi tin các bảng

Tính lại toàn bộ metric từ per-sentence events và đối chiếu **tuyệt đối** với số trong `eval_report.json` (nb3) và `zeroshot_eval_report.json` (nb3b). Nếu MISMATCH → dừng (raise), không xuất kết quả: phiên bản Input/sẻ code đã lệch, phải xử lý trước.


In [6]:
REF_RESULTS = {}
_rep3 = json.loads(Path(INPUT_FILES[REPORT_NB3]).read_text(encoding='utf-8'))
REF_RESULTS.update(_rep3.get('results') or {})
if REPORT_NB3B in INPUT_FILES:
    _repb = json.loads(Path(INPUT_FILES[REPORT_NB3B]).read_text(encoding='utf-8'))
    REF_RESULTS.update(_repb.get('results') or {})

SET_TO_REF_KEY = {
    'val_run1': 'val_run1_pure',
    'val_run2': 'val_run2_aug',
    'test_run1': 'test_run1_pure',
    'test_run2': 'test_run2_aug',
    'val_zeroshot': 'val_zeroshot',
    'test_zeroshot': 'test_zeroshot',
}


def recomputed_metrics(agg):
    return {
        'tp': agg['tp'], 'fp': agg['fp'], 'fn': agg['fn'],
        'precision': agg['precision'], 'recall': agg['recall'], 'f1': agg['f1'],
        'correct_at_tp': agg['correct_at_tp'], 'correction_accuracy': agg['corr_acc'],
        'over_correction_rate': agg['overcorr'], 'clean_tokens': agg['clean_tokens'],
        'clean_sents_total': agg['clean_total'], 'clean_sents_preserved': agg['clean_kept'],
        'clean_retention_rate': agg['clean_retention'],
    }


def ref_metrics(ref):
    d, c, o = ref['detection'], ref['correction'], ref['over_correction']
    return {
        'tp': d['tp'], 'fp': d['fp'], 'fn': d['fn'],
        'precision': d['precision'], 'recall': d['recall'], 'f1': d['f1'],
        'correct_at_tp': c['correct_at_tp'], 'correction_accuracy': c['accuracy'],
        'over_correction_rate': o['rate'], 'clean_tokens': o['clean_tokens'],
        'clean_sents_total': o['clean_sents_total'],
        'clean_sents_preserved': o['clean_sents_preserved'],
        'clean_retention_rate': o['clean_retention_rate'],
    }


CONSISTENCY = {}
_all_ok = True
print('=== CONSISTENCY CHECK (per-sentence events vs report JSON của nb3/nb3b) ===')
for set_name in sorted(ANALYSIS):
    ref_key = SET_TO_REF_KEY[set_name]
    if ref_key not in REF_RESULTS:
        CONSISTENCY[set_name] = {'status': 'skipped_no_reference'}
        print(f'  {set_name:14s}: SKIP (không có {ref_key} trong report JSON — chỉ áp dụng với zeroshot thiếu file)')
        continue
    _, agg = ANALYSIS[set_name]
    mine = recomputed_metrics(agg)
    ref = ref_metrics(REF_RESULTS[ref_key])
    diffs = {}
    for k in mine:
        if isinstance(mine[k], int) and isinstance(ref[k], int):
            if mine[k] != ref[k]:
                diffs[k] = [mine[k], ref[k]]
        elif abs(mine[k] - ref[k]) > 1e-9:
            diffs[k] = [mine[k], ref[k]]
    ok = not diffs
    _all_ok = _all_ok and ok
    CONSISTENCY[set_name] = {'status': 'ok' if ok else 'mismatch',
                             'ref_key': ref_key,
                             'diffs': diffs if not ok else None}
    print(f'  {set_name:14s}: ' + ('OK' if ok else f'MISMATCH {diffs}'))
if not _all_ok:
    raise RuntimeError(
        'CONSISTENCY CHECK FAIL — analyze_sentence không tái tạo được metrics của nb3/nb3b. '
        'KHÔNG tin các bảng phía dưới. Kiểm tra SHARED_CELLS_VERSION và phiên bản các file Input.')
print('=== CONSISTENCY CHECK PASS — số per-sentence khớp tuyệt đối với nb3/nb3b ===')


=== CONSISTENCY CHECK (per-sentence events vs report JSON của nb3/nb3b) ===
  test_run1     : OK
  test_run2     : OK
  val_run1      : OK
  val_run2      : OK
=== CONSISTENCY CHECK PASS — số per-sentence khớp tuyệt đối với nb3/nb3b ===


## §5. Các bảng phân tích T1–T7

Mỗi ô kèm **số raw (n=…)**; stratum có `n<30` được đánh dấu — không kết luận trên stratum quá nhỏ. Toàn bộ số liệu cũng được ghi vào `RESULTS` để xuất `error_analysis_report.json` ở §7.


In [7]:
RESULTS = {}


def md_table(headers, rows):
    lines = ['| ' + ' | '.join(str(h) for h in headers) + ' |',
             '|' + '|'.join(['---'] * len(headers)) + '|']
    for r in rows:
        lines.append('| ' + ' | '.join(str(c) for c in r) + ' |')
    print('\n'.join(lines))


def t1_by_cat(set_name):
    _, agg = ANALYSIS[set_name]
    out = {}
    for cat in ('nonword', 'realword'):
        st = agg['stratified'][cat]
        out[cat] = {
            'gold': st['gold'], 'detected': st['detected'], 'corrected': st['corrected'],
            'corr_acc': (st['corrected'] / st['detected']) if st['detected'] else None,
            'recall': (st['detected'] / st['gold']) if st['gold'] else None,
        }
    return out


def run_cols(split):
    return [r for r in ('run1', 'run2', 'zeroshot') if (split + '_' + r) in ANALYSIS]


RESULTS['t1'] = {set_name: t1_by_cat(set_name) for set_name in ANALYSIS}

print('### T1 — Correction Accuracy @TP theo loại lỗi (Q1) — ô: CorrAcc (đúng/detect) · rec = detection recall\n')
for split in ('test', 'val'):
    cols = run_cols(split)
    if not cols:
        continue
    rows = []
    for cat in ('nonword', 'realword'):
        cells_ = [cat]
        for r in cols:
            key = split + '_' + r
            d = RESULTS['t1'][key][cat]
            if d['corr_acc'] is None:
                acc = 'N/A'
            else:
                acc = f"{d['corr_acc']:.2%} ({d['corrected']}/{d['detected']})"
            rec_ = 'N/A' if d['recall'] is None else f"{d['recall']:.2%}"
            flag = ' *n<30' if (d['detected'] < 30 or d['gold'] < 30) else ''
            cells_.append(f'{acc} · rec {rec_}{flag}')
        rows.append(cells_)
    md_table([split.upper()] + cols, rows)
print('\n(Phân loại non-word/real-word theo bảng âm tiết 7.884 — cùng quy ước stratified của nb3.)')


### T1 — Correction Accuracy @TP theo loại lỗi (Q1) — ô: CorrAcc (đúng/detect) · rec = detection recall

| TEST | run1 | run2 |
|---|---|---|
| nonword | 65.84% (9319/14153) · rec 69.62% | 67.57% (10591/15673) · rec 77.09% |
| realword | 67.87% (2009/2960) · rec 64.98% | 69.84% (2100/3007) · rec 66.02% |
| VAL | run1 | run2 |
|---|---|---|
| nonword | 85.71% (192/224) · rec 75.68% | 89.09% (196/220) · rec 74.32% |
| realword | 84.39% (508/602) · rec 71.07% | 85.57% (504/589) · rec 69.54% |

(Phân loại non-word/real-word theo bảng âm tiết 7.884 — cùng quy ước stratified của nb3.)


In [8]:
def t2_recall_by_density(set_name):
    _, agg = ANALYSIS[set_name]
    tp_c = collections.Counter(e['density_bin'] for e in agg['tp_events'])
    fn_c = collections.Counter(e['density_bin'] for e in agg['fn_events'])
    out = {}
    for b in DENSITY_BINS:
        gold = tp_c[b] + fn_c[b]
        out[b] = {'gold': gold, 'tp': tp_c[b], 'recall': (tp_c[b] / gold) if gold else None}
    return out


RESULTS['t2'] = {s: t2_recall_by_density(s) for s in ANALYSIS if s.startswith('test')}

cols = run_cols('test')
print('### T2 — Detection recall theo mật độ lỗi/câu (số gold block, TEST) (Q2)\n')
rows = []
for b in DENSITY_BINS:
    row = [b + ' lỗi/câu']
    for r in cols:
        key = 'test_' + r
        d = RESULTS['t2'][key][b]
        row.append('N/A' if d['recall'] is None else f"{d['recall']:.2%} (n={d['gold']})")
    rows.append(row)
md_table(['Mật độ'] + cols, rows)


### T2 — Detection recall theo mật độ lỗi/câu (số gold block, TEST) (Q2)

| Mật độ | run1 | run2 |
|---|---|---|
| 1 lỗi/câu | 67.75% (n=865) | 70.87% (n=865) |
| 2 lỗi/câu | 68.73% (n=2251) | 73.12% (n=2251) |
| 3 lỗi/câu | 69.87% (n=3213) | 75.10% (n=3213) |
| >=4 lỗi/câu | 68.63% (n=18556) | 75.49% (n=18556) |


In [9]:
def t3_fp_breakdown(set_name):
    _, agg = ANALYSIS[set_name]
    by_type = collections.Counter(e['block_type'] for e in agg['fp_events'])
    top_pred = collections.Counter(e['pred_str'] for e in agg['fp_events']
                                   if e['block_type'] != 'delete').most_common(15)
    top_del = collections.Counter(e['src_tok'] for e in agg['fp_events']
                                  if e['block_type'] == 'delete').most_common(15)
    return {
        'total_fp_positions': len(agg['fp_events']),
        'by_type': dict(by_type),
        'top_pred_targets_non_delete': top_pred,
        'top_deleted_src_tokens': top_del,
    }


RESULTS['t3'] = {s: t3_fp_breakdown(s) for s in ANALYSIS}

all_types = ['delete', 'substitute', 'insert', 'split', 'merge', 'multi']
cols = sorted(ANALYSIS)
print('### T3 — FP breakdown theo block type (Q3) — ô: số vị trí (% trên tổng FP của set)\n')
rows = []
for t in all_types:
    row = [t]
    for s in cols:
        d = RESULTS['t3'][s]
        c = d['by_type'].get(t, 0)
        tot = d['total_fp_positions']
        row.append(f'{c} ({c / tot:.1%})' if tot else 'N/A')
    rows.append(row)
row = ['TỔNG FP positions']
for s in cols:
    row.append(str(RESULTS['t3'][s]['total_fp_positions']))
rows.append(row)
md_table(['Block type'] + cols, rows)

if 'test_run2' in RESULTS['t3']:
    print('\nTop-15 token BỊ SỬA VÀO ở FP non-delete (TEST, Run 2):')
    for tok, c in RESULTS['t3']['test_run2']['top_pred_targets_non_delete'][:15]:
        print(f'  {tok!r:26s} x {c}')
    print('\nTop-15 token BỊ XÓA ở FP delete (TEST, Run 2):')
    for tok, c in RESULTS['t3']['test_run2']['top_deleted_src_tokens'][:15]:
        print(f'  {tok!r:26s} x {c}')


### T3 — FP breakdown theo block type (Q3) — ô: số vị trí (% trên tổng FP của set)

| Block type | test_run1 | test_run2 | val_run1 | val_run2 |
|---|---|---|---|---|
| delete | 1507 (54.2%) | 1304 (45.4%) | 143 (71.5%) | 147 (68.4%) |
| substitute | 86 (3.1%) | 103 (3.6%) | 27 (13.5%) | 32 (14.9%) |
| insert | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) | 0 (0.0%) |
| split | 8 (0.3%) | 5 (0.2%) | 1 (0.5%) | 1 (0.5%) |
| merge | 755 (27.2%) | 785 (27.3%) | 25 (12.5%) | 28 (13.0%) |
| multi | 424 (15.3%) | 675 (23.5%) | 4 (2.0%) | 7 (3.3%) |
| TỔNG FP positions | 2780 | 2872 | 200 | 215 |

Top-15 token BỊ SỬA VÀO ở FP non-delete (TEST, Run 2):
  'F Paris'                  x 216
  'Nguwo'                    x 95
  'Đ'                        x 60
  'thành'                    x 41
  'đủ ...'                   x 28
  'Trưng'                    x 22
  'cho buổi dạo chơi cuối'   x 13
  'sự nỗ lực phấn đấu của cán bộ , công chức ngành' x 12
  'Hồng'                     x 12
  'hạt dưa hấu gieo lên lưng tr

In [10]:
def t4_wrong_tp_taxonomy(set_name):
    _, agg = ANALYSIS[set_name]
    wrong = [e for e in agg['tp_events'] if not e['correct']]
    tax = collections.Counter()
    for e in wrong:
        pt = e['pred_tgt']
        toks = pt.split(' ')
        if pt == '':
            tax['deletion'] += 1
        elif all(is_punct_token(t) for t in toks):
            tax['punct_tgt'] += 1
        elif pt.lower() == (e['src_tok'] or '').lower():
            tax['same_as_src'] += 1
        elif all(t.lower() in SYLL_SET for t in toks):
            tax['valid_syllable_tgt'] += 1
        else:
            tax['nonword_tgt'] += 1
    return {'n_tp': len(agg['tp_events']), 'n_wrong': len(wrong), 'taxonomy': dict(tax)}


RESULTS['t4'] = {s: t4_wrong_tp_taxonomy(s) for s in ANALYSIS}

print('### T4 — Sửa SAI nội dung ở vị trí detect đúng (TP): model sửa thành gì? (Q4)\n')
labels_ = ['deletion (tgt rỗng)', 'same_as_src', 'punct_tgt', 'valid_syllable_tgt', 'nonword_tgt']
rows = []
for s in sorted(ANALYSIS):
    d = RESULTS['t4'][s]
    tot = d['n_wrong']
    for lab in labels_:
        key = lab.split(' ')[0]
        c = d['taxonomy'].get(key, 0)
        rows.append([s, lab, f'{c} ({c / tot:.1%})' if tot else 'N/A'])
    rows.append([s, 'TỔNG sai', f"{tot}/{d['n_tp']} TP"])
md_table(['Set', 'Nhãn', 'Số & %'], rows)


### T4 — Sửa SAI nội dung ở vị trí detect đúng (TP): model sửa thành gì? (Q4)

| Set | Nhãn | Số & % |
|---|---|---|
| test_run1 | deletion (tgt rỗng) | 805 (13.9%) |
| test_run1 | same_as_src | 12 (0.2%) |
| test_run1 | punct_tgt | 7 (0.1%) |
| test_run1 | valid_syllable_tgt | 4101 (70.9%) |
| test_run1 | nonword_tgt | 860 (14.9%) |
| test_run1 | TỔNG sai | 5785/17113 TP |
| test_run2 | deletion (tgt rỗng) | 693 (11.6%) |
| test_run2 | same_as_src | 12 (0.2%) |
| test_run2 | punct_tgt | 19 (0.3%) |
| test_run2 | valid_syllable_tgt | 4622 (77.2%) |
| test_run2 | nonword_tgt | 643 (10.7%) |
| test_run2 | TỔNG sai | 5989/18680 TP |
| val_run1 | deletion (tgt rỗng) | 28 (22.2%) |
| val_run1 | same_as_src | 0 (0.0%) |
| val_run1 | punct_tgt | 1 (0.8%) |
| val_run1 | valid_syllable_tgt | 92 (73.0%) |
| val_run1 | nonword_tgt | 5 (4.0%) |
| val_run1 | TỔNG sai | 126/826 TP |
| val_run2 | deletion (tgt rỗng) | 23 (21.1%) |
| val_run2 | same_as_src | 0 (0.0%) |
| val_run2 | punct_tgt | 1 (0.9%

In [11]:
def t5_fn_anatomy(set_name):
    _, agg = ANALYSIS[set_name]
    fns = agg['fn_events']
    n = len(fns)
    pure = sum(1 for e in fns if e['model_edit_count_in_sent'] == 0)
    edited = [e for e in fns if e['model_edit_count_in_sent'] > 0]
    dists = [e['min_dist_to_model_edit'] for e in edited if e['min_dist_to_model_edit'] is not None]
    bk = collections.Counter()
    for d in dists:
        if d <= 2:
            bk['1-2'] += 1
        elif d <= 5:
            bk['3-5'] += 1
        else:
            bk['>5'] += 1
    return {
        'n_fn': n, 'pure_copy': pure, 'edited_elsewhere': len(dists),
        'dist_buckets': dict(bk),
        'mean_min_dist': (sum(dists) / len(dists)) if dists else None,
    }


RESULTS['t5'] = {s: t5_fn_anatomy(s) for s in ANALYSIS if s.startswith('test')}

print('### T5 — FN anatomy (TEST, Q5): model bỏ sót kiểu gì?\n')


def _pct_or_na(num, den):
    return f'{num / den:.1%}' if den else 'N/A'


rows = []
for s in sorted(RESULTS['t5']):
    d = RESULTS['t5'][s]
    mean_s = 'N/A' if d['mean_min_dist'] is None else f"{d['mean_min_dist']:.2f}"
    rows.append([
        s,
        f"{d['pure_copy']} ({_pct_or_na(d['pure_copy'], d['n_fn'])})",
        f"{d['edited_elsewhere']} ({_pct_or_na(d['edited_elsewhere'], d['n_fn'])})",
        str(d['dist_buckets']),
        mean_s,
    ])
md_table(['Set', 'FN copy thuần (câu 0 edit)', 'FN khi câu có edit', 'Khoảng cách tới edit gần nhất', 'K.cách TB'], rows)
print('\n(pure copy = model xuất nguyên câu nguồn; edited_elsewhere = model có sửa nhưng lệch vị trí.)')


### T5 — FN anatomy (TEST, Q5): model bỏ sót kiểu gì?

| Set | FN copy thuần (câu 0 edit) | FN khi câu có edit | Khoảng cách tới edit gần nhất | K.cách TB |
|---|---|---|---|---|
| test_run1 | 593 (7.6%) | 7144 (91.9%) | {'>5': 3278, '1-2': 1736, '3-5': 2130} | 7.18 |
| test_run2 | 472 (7.6%) | 5722 (92.2%) | {'1-2': 1541, '3-5': 1759, '>5': 2422} | 6.51 |

(pure copy = model xuất nguyên câu nguồn; edited_elsewhere = model có sửa nhưng lệch vị trí.)


In [12]:
def t6_by_bucket(set_name, key, order):
    _, agg = ANALYSIS[set_name]
    tp_c = collections.Counter()
    fn_c = collections.Counter()
    corr_c = collections.Counter()
    for e in agg['fn_events']:
        fn_c[e[key]] += 1
    for e in agg['tp_events']:
        tp_c[e[key]] += 1
        if e['correct']:
            corr_c[e[key]] += 1
    out = {}
    for b in order:
        gold = tp_c[b] + fn_c[b]
        out[b] = {
            'gold': gold, 'tp': tp_c[b],
            'recall': (tp_c[b] / gold) if gold else None,
            'corr_acc': (corr_c[b] / tp_c[b]) if tp_c[b] else None,
        }
    return out


RELPOS_ORDER = [b[0] for b in RELPOS_BUCKETS]
LEN_ORDER = [b[0] for b in LEN_BUCKETS]
RESULTS['t6_relpos'] = {s: t6_by_bucket(s, 'rel_pos', RELPOS_ORDER) for s in ANALYSIS if s.startswith('test')}
RESULTS['t6_len'] = {s: t6_by_bucket(s, 'len_bucket', LEN_ORDER) for s in ANALYSIS if s.startswith('test')}

for title, map_key, order in [('vị trí tương đối trong câu', 't6_relpos', RELPOS_ORDER),
                              ('độ dài câu (token nguồn)', 't6_len', LEN_ORDER)]:
    print(f'### T6 — Recall & CorrAcc theo {title} (TEST, Q6)\n')
    cols = sorted(RESULTS[map_key])
    rows = []
    for b in order:
        row = [b]
        for s in cols:
            d = RESULTS[map_key][s][b]
            rec_ = 'N/A' if d['recall'] is None else f"{d['recall']:.2%}"
            ca = 'N/A' if d['corr_acc'] is None else f"{d['corr_acc']:.2%}"
            row.append(f'rec {rec_} · acc {ca} (n={d["gold"]})')
        rows.append(row)
    md_table(['Bucket'] + cols, rows)
    print()


### T6 — Recall & CorrAcc theo vị trí tương đối trong câu (TEST, Q6)

| Bucket | test_run1 | test_run2 |
|---|---|---|
| 0-25% | rec 69.57% · acc 67.04% (n=6808) | rec 74.65% · acc 68.69% (n=6808) |
| 25-50% | rec 70.46% · acc 67.12% (n=6212) | rec 76.35% · acc 69.41% (n=6212) |
| 50-75% | rec 68.43% · acc 65.20% (n=6466) | rec 75.19% · acc 66.52% (n=6466) |
| 75-100% | rec 66.22% · acc 65.17% (n=5399) | rec 73.96% · acc 66.97% (n=5399) |

### T6 — Recall & CorrAcc theo độ dài câu (token nguồn) (TEST, Q6)

| Bucket | test_run1 | test_run2 |
|---|---|---|
| <15 | rec 64.22% · acc 57.79% (n=1269) | rec 68.48% · acc 59.61% (n=1269) |
| 15-30 | rec 70.68% · acc 64.01% (n=6023) | rec 75.15% · acc 65.86% (n=6023) |
| 30-50 | rec 70.05% · acc 67.34% (n=9096) | rec 76.34% · acc 68.82% (n=9096) |
| >=50 | rec 66.72% · acc 67.75% (n=8497) | rec 74.63% · acc 69.59% (n=8497) |



In [13]:
def t7_fp_on_clean(set_name):
    _, agg = ANALYSIS[set_name]
    fp_clean = [e for e in agg['fp_events'] if e['sentence_clean']]
    sents = sorted({e['sent_id'] for e in fp_clean})
    by_type = collections.Counter(e['block_type'] for e in fp_clean)
    return {
        'n_clean_total': agg['clean_total'],
        'n_clean_broken': len(sents),
        'n_fp_positions': len(fp_clean),
        'by_type': dict(by_type),
        'broken_sent_ids_sample': sents[:10],
    }


RESULTS['t7'] = {s: t7_fp_on_clean(s) for s in ANALYSIS if s.startswith('test')}

print('### T7 — FP trên câu sạch TEST: nguyên nhân phá Clean Retention\n')
rows = []
for s in sorted(RESULTS['t7']):
    d = RESULTS['t7'][s]
    rows.append([s, f"{d['n_clean_broken']}/{d['n_clean_total']}", str(d['n_fp_positions']), str(d['by_type'])])
md_table(['Set', 'Câu sạch bị phá', 'FP positions', 'Theo block type'], rows)

if 'test_run2' in ANALYSIS:
    records, preds = REC_PREDS['test_run2']
    print('\n5 ví dụ đầu — câu sạch bị Run 2 phá (TEST):')
    seen = set()
    for e in ANALYSIS['test_run2'][1]['fp_events']:
        if not e['sentence_clean'] or e['sent_id'] in seen:
            continue
        seen.add(e['sent_id'])
        print(f'  [idx {e["sent_id"]}] SRC : {records[e["sent_id"]]["text"]}')
        print(f'             PRED: {preds[e["sent_id"]]}')
        if len(seen) >= 5:
            break


### T7 — FP trên câu sạch TEST: nguyên nhân phá Clean Retention

| Set | Câu sạch bị phá | FP positions | Theo block type |
|---|---|---|---|
| test_run1 | 87/653 | 120 | {'delete': 102, 'merge': 2, 'split': 1, 'substitute': 13, 'multi': 2} |
| test_run2 | 92/653 | 135 | {'merge': 12, 'delete': 106, 'split': 1, 'substitute': 14, 'multi': 2} |

5 ví dụ đầu — câu sạch bị Run 2 phá (TEST):
  [idx 148] SRC : Chẳng hạn Gonlag, là một trong hàng ngàn chiến binh thế hệ mới theo phong trào gọi là hacktivisme, ghép từ hacking - tin tặc và activisme - chiến binh.
             PRED: Chẳng hạn Gonlag, là một trong hàng ngàn chiến binh thế hệ mới theo phong trào gọi là hacktivisme, ghép từ hacking - tin tặc vàisme - chiến binh.
  [idx 174] SRC : Cũng theo nghị quyết này, tại kỳ họp QH, sẽ rút ngắn thời gian trình bày các báo cáo tại hội trường xuống còn 15 đến 20 phút, trừ báo cáo của Chính phủ, báo cáo thẩm tra về kinh tế-xã hội, ngân sách.
             PRED: Cũng theo nghị quyết này, tại kỳ họp Q

## §6. Adjudication 100 câu suspect — ước lượng nhiễu pseudo-gold (Q7)

**Rubric nhãn (điền `label` ∈ {A,B,C,D} vào `adjudication_samples.json`):**
- **A** — hai vế song song, các edit là lỗi chính tả thật → pseudo-gold OK.
- **B** — phía source đúng / bản sửa sai → pseudo-gold **NHIỄU** (model bị phạt oan).
- **C** — edit phi chính tả (số liệu, viết tắt, tên riêng, format) — đếm riêng, **không** tính là nhiễu.
- **D** — hai vế không song song / align hỏng → tính là nhiễu.

**Quy trình:** tải `adjudication_samples.json` → điền 100 nhãn → đổi tên `adjudication_filled.json` → upload làm Input → chạy lại notebook từ đầu. Cell sau tự nhận diện file và tính: tỷ lệ nhiễu (B+D) + Wilson CI 95% + **sensitivity** (metric tính lại trên 100 câu được adjudicate — so sánh subset suspect, không phải exclusion toàn test). Ngưỡng pre-registered: nhiễu > 8% → mọi chỉ số trên TEST phải đọc kèm caveat.


In [14]:
suspect_idx = [i for i, r in enumerate(test_records) if r.get('suspect')]
strata_pool = {name: [] for name, lo, hi in ADJ_RATIO_STRATA}
for i in suspect_idx:
    ratio = test_records[i].get('edit_ratio') or 0.0
    for name, lo, hi in ADJ_RATIO_STRATA:
        if lo <= ratio < hi:
            strata_pool[name].append(i)
            break

total_sus = sum(len(p) for p in strata_pool.values())
n_take = min(ADJ_N, total_sus)
quota_frac = {name: (len(p) * n_take / total_sus) if total_sus else 0.0 for name, p in strata_pool.items()}
quota_base = {name: int(math.floor(q)) for name, q in quota_frac.items()}
remainder = n_take - sum(quota_base.values())
order_by_frac = sorted(quota_frac, key=lambda k: quota_frac[k] - quota_base[k], reverse=True)
for k in order_by_frac[:remainder]:
    quota_base[k] += 1

rng = random.Random(SEED)
adj_picked = []
for name, pool in strata_pool.items():
    for i in rng.sample(pool, min(quota_base[name], len(pool))):
        adj_picked.append((name, i))
adj_picked.sort(key=lambda x: x[1])


def render_marked(tokens, blocks, side):
    marked = set()
    for b in blocks:
        lo, hi = b['src_span'] if side == 'src' else b['tgt_span']
        marked.update(range(lo, hi))
    return ' '.join('[' + t + ']' if i in marked else t for i, t in enumerate(tokens))


RUBRIC = {
    'A': 'Hai vế song song, các edit là lỗi chính tả thật → pseudo-gold OK',
    'B': 'Phía source đúng / bản sửa sai → pseudo-gold NHIỄU (model bị phạt oan)',
    'C': 'Edit phi chính tả (số liệu, viết tắt, tên riêng, format) — không tính là nhiễu',
    'D': 'Hai vế không song song / align hỏng → tính là nhiễu',
}

adj_records = []
for name, i in adj_picked:
    r = test_records[i]
    adj_records.append({
        'sample_id': 'adj_' + format(i, '05d'),
        'stratum': name,
        'test_index': i,
        'text': r['text'],
        'corrected_text': r['corrected_text'],
        'src_marked': render_marked(r['src_tokens'], r['edit_blocks'], 'src'),
        'tgt_marked': render_marked(r['tgt_tokens'], r['edit_blocks'], 'tgt'),
        'edit_blocks': [{
            'type': b['type'], 'position': b['position'],
            'src': ' '.join(b['src_tokens']) or '∅',
            'tgt': ' '.join(b['tgt_tokens']) or '∅',
            'punct_only': b['punct_only'],
        } for b in r['edit_blocks']],
        'edit_ratio': r.get('edit_ratio'),
        'n_src_tokens': len(r['src_tokens']),
        'label': None,
        'label_note': '',
    })

adj_payload = {
    'created': RUN_STAMP,
    'notebook': 'nb4_error_analysis',
    'seed': SEED,
    'n_samples': len(adj_records),
    'strata_counts': dict(collections.Counter(s['stratum'] for s in adj_records)),
    'rubric': RUBRIC,
    'instructions': [
        'Điền label in {A,B,C,D} vào trường label của mỗi mẫu (rubric ở trên); ghi chú thêm vào label_note nếu cần.',
        'Đổi tên file thành adjudication_filled.json (giữ nguyên schema) và upload làm Input cho notebook này.',
        'Chạy lại notebook từ đầu — cell adjudication sẽ tự nhận diện và tính tỷ lệ nhiễu + sensitivity.',
    ],
    'samples': adj_records,
}

with open(OUTPUT_DIR / 'adjudication_samples.json', 'w', encoding='utf-8') as f:
    json.dump(adj_payload, f, ensure_ascii=False, indent=2)

print(f'Đã xuất adjudication_samples.json: {len(adj_records)} mẫu (mục tiêu {ADJ_N}, seed {SEED}).')
print('Phân bổ strata:', dict(collections.Counter(s['stratum'] for s in adj_records)),
      '| pool suspect:', dict(collections.Counter({k: len(v) for k, v in strata_pool.items()})))
for s in adj_records[:2]:
    print(f"\n—— {s['sample_id']} · stratum={s['stratum']} · edit_ratio={s['edit_ratio']}")
    print('  SRC:', s['src_marked'])
    print('  TGT:', s['tgt_marked'])


Đã xuất adjudication_samples.json: 100 mẫu (mục tiêu 100, seed 42).
Phân bổ strata: {'0.3-0.4': 88, '0.4-0.5': 10, '>=0.5': 2} | pool suspect: {'0.3-0.4': 2035, '0.4-0.5': 227, '>=0.5': 54}

—— adj_00044 · stratum=0.3-0.4 · edit_ratio=0.3571
  SRC: Một điều [lon] lưu ý là người chơi có thể [thaw] đổi nhóm tùy thích nhưng sẽ không [đưac] [thak] gia [thk] đấu hai trận liên tiếp .
  TGT: Một điều [cần] lưu ý là người chơi có thể [thay] đổi nhóm tùy thích nhưng sẽ không [được] [tham] gia [thi] đấu hai trận liên tiếp .

—— adj_00140 · stratum=0.3-0.4 · edit_ratio=0.3448
  SRC: Điều đáng tiếc là U23 Việt Nam đã [nbỏ] lỡ quá [nhiềh] [vơ] hội , mong [rằnjg] ở các trận đấu sau các tiền đạo sẽ có [duygên] hơn
  TGT: Điều đáng tiếc là U23 Việt Nam đã [bỏ] lỡ quá [nhiều] [cơ] hội , mong [rằng] ở các trận đấu sau các tiền đạo sẽ có [duyên] hơn


In [15]:
filled_path = INPUT_FILES.get('adjudication_filled.json')
if filled_path is None:
    ADJUDICATION = {'status': 'pending_manual_labels'}
    print('Chưa có adjudication_filled.json (tùy chọn) — bỏ qua phân tích adjudication.')
    print('Quy trình: tải adjudication_samples.json → điền label A/B/C/D cho đủ mẫu → '
          'đổi tên adjudication_filled.json → upload làm Input → chạy lại notebook từ đầu.')
else:
    filled = json.loads(Path(filled_path).read_text(encoding='utf-8'))
    filled_samples = filled.get('samples') if isinstance(filled, dict) else filled
    by_id = {s['sample_id']: s for s in filled_samples}
    labels = {}
    for rec in adj_records:
        src_ = by_id.get(rec['sample_id'])
        if src_ is None:
            raise ValueError('Thiếu nhãn cho ' + rec['sample_id'] + ' trong adjudication_filled.json')
        lab = src_.get('label')
        if lab not in RUBRIC:
            raise ValueError(f'Nhãn không hợp lệ cho {rec["sample_id"]}: {lab!r} (phải A/B/C/D)')
        labels[rec['sample_id']] = lab

    cnt = collections.Counter(labels.values())
    n_lbl = sum(cnt.values())
    noise = (cnt['B'] + cnt['D']) / n_lbl if n_lbl else None
    z = 1.959963984540054
    if noise is not None:
        denom = 1 + z * z / n_lbl
        center = (noise + z * z / (2 * n_lbl)) / denom
        half = z * math.sqrt(noise * (1 - noise) / n_lbl + z * z / (4 * n_lbl * n_lbl)) / denom
        ci = [max(0.0, center - half), min(1.0, center + half)]
    else:
        ci = None

    keep_set = {rec['test_index'] for rec in adj_records if labels[rec['sample_id']] not in ('B', 'D')}
    sensitivity = {}
    for set_name in [s for s in ('test_run1', 'test_run2', 'test_zeroshot') if s in ANALYSIS]:
        analyzed, full = ANALYSIS[set_name]
        kept = [a for i, a in enumerate(analyzed) if i in keep_set]
        agg = aggregate_sets(kept)
        by_cat = {}
        for c in ('nonword', 'realword'):
            st_k = agg['stratified'][c]
            st_f = full['stratified'][c]
            by_cat[c] = {
                'corr_acc_kept': (st_k['corrected'] / st_k['detected']) if st_k['detected'] else None,
                'corr_acc_full': (st_f['corrected'] / st_f['detected']) if st_f['detected'] else None,
            }
        sensitivity[set_name] = {
            'n_sents_kept': len(kept), 'n_sents_full': len(analyzed),
            'f1_kept': agg['f1'], 'f1_full': full['f1'],
            'corr_acc_kept': agg['corr_acc'], 'corr_acc_full': full['corr_acc'],
            'overcorr_kept': agg['overcorr'], 'overcorr_full': full['overcorr'],
            'by_cat': by_cat,
        }

    ADJUDICATION = {
        'status': 'done', 'n': n_lbl, 'label_counts': dict(cnt),
        'noise_rate_B_plus_D': noise, 'wilson95_ci': ci,
        'threshold_caveat': 'noise > 8% => moi chi so tren TEST phai doc kem caveat',
        'per_stratum': {
            name: {
                'n': sum(1 for r in adj_records if r['stratum'] == name),
                'noise': sum(1 for r in adj_records
                             if r['stratum'] == name and labels[r['sample_id']] in ('B', 'D')),
            }
            for name, _lo, _hi in ADJ_RATIO_STRATA
        },
        'sensitivity': sensitivity,
    }

    print('Phân bổ nhãn:', dict(cnt))
    if noise is None:
        print('Không có nhãn hợp lệ nào (n=0) — bỏ qua ước lượng nhiễu + Wilson CI.')
    else:
        print(f'Tỷ lệ nhiễu pseudo-gold (B+D): {noise:.1%} · Wilson 95% CI: [{ci[0]:.1%}, {ci[1]:.1%}] (n={n_lbl})')
    for set_name, s_ in sensitivity.items():
        print(f'  {set_name}: F1 {s_["f1_full"]:.2%} → {s_["f1_kept"]:.2%} '
              f'(metric tính trên {s_["n_sents_kept"]} câu được adjudicate — không phải toàn TEST) · '
              f'CorrAcc {s_["corr_acc_full"]:.2%} → {s_["corr_acc_kept"]:.2%}')
    if noise is not None and noise > 0.08:
        print('CẢNH BÁO: nhiễu pseudo-gold > 8% → mọi chỉ số trên TEST phải đọc kèm caveat.')


Phân bổ nhãn: {'A': 99, 'C': 1}
Tỷ lệ nhiễu pseudo-gold (B+D): 0.0% · Wilson 95% CI: [0.0%, 3.7%] (n=100)
  test_run1: F1 76.43% → 78.29% (loại 5883 câu B/D) · CorrAcc 66.20% → 75.00%
  test_run2: F1 80.45% → 82.13% (loại 5883 câu B/D) · CorrAcc 67.94% → 74.71%


## §7. Xuất `error_analysis_report.json` + tóm tắt REPORT-ready

File chứa: config, inputs, consistency check, toàn bộ T1–T7, adjudication. Khối tóm tắt cuối in ra dạng Markdown để dán vào REPORT.md (việc cập nhật REPORT.md là bước riêng, chờ duyệt).


In [16]:
def _json_default(o):
    if isinstance(o, collections.Counter):
        return dict(o)
    if isinstance(o, tuple):
        return list(o)
    raise TypeError('Không serialize được JSON: ' + str(type(o)))


error_analysis_report = {
    'created': RUN_STAMP,
    'notebook': 'nb4_error_analysis',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {
        'seed': SEED,
        'adj_n': ADJ_N,
        'density_bins': DENSITY_BINS,
        'len_buckets': [b[0] for b in LEN_BUCKETS],
        'relpos_buckets': [b[0] for b in RELPOS_BUCKETS],
        'adj_ratio_strata': [s[0] for s in ADJ_RATIO_STRATA],
        'training': False,
        'device': 'eval-only, CPU-compatible',
    },
    'inputs': {k: str(v) for k, v in sorted(INPUT_FILES.items())},
    'consistency_check': CONSISTENCY,
    'results': RESULTS,
    'adjudication': ADJUDICATION,
}

with open(OUTPUT_DIR / 'error_analysis_report.json', 'w', encoding='utf-8') as f:
    json.dump(error_analysis_report, f, ensure_ascii=False, indent=2, default=_json_default)

print('=== HOÀN TẤT XUẤT ĐẦU RA ===')
print('Các file đã ghi vào', OUTPUT_DIR)
for fn in ['error_analysis_report.json', 'adjudication_samples.json']:
    print(' -', fn)


=== HOÀN TẤT XUẤT ĐẦU RA ===
Các file đã ghi vào /kaggle/working
 - error_analysis_report.json
 - adjudication_samples.json


In [17]:
# Tóm tắt REPORT-ready — các con số chính tự điền từ RESULTS/ANALYSIS
def _overall(set_name, field):
    if set_name not in ANALYSIS:
        return 'N/A'
    return f'{ANALYSIS[set_name][1][field]:.2%}'


def _t1(set_name, cat):
    d = RESULTS.get('t1', {}).get(set_name, {}).get(cat)
    if not d or d['corr_acc'] is None:
        return 'N/A'
    return f"{d['corr_acc']:.2%} ({d['corrected']}/{d['detected']})"


def _t2(run, b):
    d = RESULTS.get('t2', {}).get('test_' + run, {}).get(b)
    if not d or d['recall'] is None:
        return 'N/A'
    return f"{d['recall']:.2%} (n={d['gold']})"


def _t3_share(set_name, btype):
    d = RESULTS.get('t3', {}).get(set_name)
    if not d or not d['total_fp_positions']:
        return 'N/A'
    c = d['by_type'].get(btype, 0)
    return f'{c} ({c / d["total_fp_positions"]:.1%})'


def _t4_share(set_name, key):
    d = RESULTS.get('t4', {}).get(set_name)
    if not d or not d['n_wrong']:
        return 'N/A'
    c = d['taxonomy'].get(key, 0)
    return f'{c} ({c / d["n_wrong"]:.1%})'


def _t5(set_name):
    d = RESULTS.get('t5', {}).get(set_name)
    if not d or not d['n_fn']:
        return 'N/A'
    return f"{d['pure_copy']} ({d['pure_copy'] / d['n_fn']:.1%})"


print('## nb4 — Error analysis: tóm tắt (copy vào REPORT.md sau khi duyệt)\n')
print(f'- **Q1 · CorrAcc @TP trên TEST (Run 2)**: tổng {_overall("test_run2", "corr_acc")} — '
      f'nonword {_t1("test_run2", "nonword")} · realword {_t1("test_run2", "realword")} '
      f'(VAL đối chiếu Run 2: nonword {_t1("val_run2", "nonword")} · realword {_t1("val_run2", "realword")})')
print(f'- **Q2 · Recall theo mật độ lỗi (TEST Run 2)**: 1 lỗi/câu {_t2("run2", "1")} · '
      f'2 lỗi {_t2("run2", "2")} · 3 lỗi {_t2("run2", "3")} · >=4 lỗi {_t2("run2", ">=4")}')
print(f'- **Q3 · FP deletion (TEST Run 2)**: {_t3_share("test_run2", "delete")} trong tổng FP — '
      f'Run 1: {_t3_share("test_run1", "delete")}')
print(f'- **Q4 · TP sửa sai (TEST Run 2)**: deletion {_t4_share("test_run2", "deletion")} · '
      f'same_as_src {_t4_share("test_run2", "same_as_src")} · '
      f'valid_syllable {_t4_share("test_run2", "valid_syllable_tgt")} · '
      f'nonword_tgt {_t4_share("test_run2", "nonword_tgt")}')
print(f'- **Q5 · FN copy thuần (TEST Run 2)**: {_t5("test_run2")} — Run 1: {_t5("test_run1")}')
adj_status = ADJUDICATION.get('status') if isinstance(ADJUDICATION, dict) else None
if adj_status == 'done':
    print(f"- **Q7 · Adjudication**: nhiễu pseudo-gold (B+D) = {ADJUDICATION['noise_rate_B_plus_D']:.1%} "
          f"(CI95 [{ADJUDICATION['wilson95_ci'][0]:.1%}, {ADJUDICATION['wilson95_ci'][1]:.1%}], n={ADJUDICATION['n']})")
else:
    print('- **Q7 · Adjudication**: CHƯA có nhãn — điền adjudication_filled.json rồi chạy lại.')
print('\n(Chi tiết đầy đủ: error_analysis_report.json · số đã đối chiếu tuyệt đối với nb3/nb3b ở §4.)')


## nb4 — Error analysis: tóm tắt (copy vào REPORT.md sau khi duyệt)

- **Q1 · CorrAcc @TP trên TEST (Run 2)**: tổng 67.94% — nonword 67.57% (10591/15673) · realword 69.84% (2100/3007) (VAL đối chiếu Run 2: nonword 89.09% (196/220) · realword 85.57% (504/589))
- **Q2 · Recall theo mật độ lỗi (TEST Run 2)**: 1 lỗi/câu 70.87% (n=865) · 2 lỗi 73.12% (n=2251) · 3 lỗi 75.10% (n=3213) · >=4 lỗi 75.49% (n=18556)
- **Q3 · FP deletion (TEST Run 2)**: 1304 (45.4%) trong tổng FP — Run 1: 1507 (54.2%)
- **Q4 · TP sửa sai (TEST Run 2)**: deletion 693 (11.6%) · same_as_src 12 (0.2%) · valid_syllable 4622 (77.2%) · nonword_tgt 643 (10.7%)
- **Q5 · FN copy thuần (TEST Run 2)**: 472 (7.6%) — Run 1: 593 (7.6%)
- **Q7 · Adjudication**: nhiễu pseudo-gold (B+D) = 0.0% (CI95 [0.0%, 3.7%], n=100)

(Chi tiết đầy đủ: error_analysis_report.json · số đã đối chiếu tuyệt đối với nb3/nb3b ở §4.)
